# 11 · F.R.I.D.A.Y. watches the whole conversation

Chapter 10 scored single answers. Real problems hide in **multi-turn** behaviour: an agent that repeats the same failing action three times can score well on every individual turn. This chapter evaluates:

| Level | Data | Source type |
|---|---|---|
| **Conversation dataset** | JSONL with a `messages` array | `jsonl` |
| **Deployed conversations** 🧪 | real conversations in Application Insights, by conversation id | `azure_ai_trace_data_source_preview` + `conversation_id_source` |
| **Sampled live traffic** 🧪 | recent conversations of an agent | `trace_source: agent_filter` |

> 🧪 Trace evaluation is **preview** and needs a recent SDK (the docs name `azure-ai-projects==2.7.0` for it). Tracing must be on (chapter 03) and Application Insights ingestion takes a few minutes. Deep dive: [docs/datasets-evals.md](../docs/datasets-evals.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

import time
oai = openai_client()
whole_conversation = {"messages": "{{item.messages}}"}   # conversation-level evaluators read all the messages at once

## 1 · Conversation dataset
Three prepared conversations: a good one, a *frustrating* one (the agent repeats itself) and a trivial one.

In [ ]:
print((ROOT / "data" / "conversations.jsonl").read_text()[:600], "...")
ds = project().datasets.upload_file(name="stark-conversations", version="1", file_path=str(ROOT / "data" / "conversations.jsonl"))

cfg_conv = item_schema({"messages": {"type": "array"}}, required=["messages"], sample=False)
criteria = [
    criterion("customer_satisfaction", "customer_satisfaction", whole_conversation),
    criterion("task_completion", "task_completion", whole_conversation),
    criterion("coherence", "coherence", whole_conversation),
]
e = oai.evals.create(name="stark-conversations", data_source_config=cfg_conv, testing_criteria=criteria)
r = oai.evals.runs.create(eval_id=e.id, name="prepared-conversations",
                          data_source={"type": "jsonl", "source": {"type": "file_id", "id": ds.id}},
                          extra_body={"evaluation_level": "conversation"})
r = wait_eval_run(e.id, r.id, oai)
print(pass_rates(r))
eval_scores(e.id, r.id, oai)

Look at the **second** conversation: a turn-level judge sees polite answers; the conversation-level judge sees an unresolved, repeated failure.

## 2 · Create real, traced conversations
Agent calls are traced **server-side** (chapter 04): Foundry writes the spans to Application Insights, so this chapter needs no `configure_azure_monitor`. Create an agent and two short conversations with it.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

AGENT = "jarvis-nb11"
agent = project().agents.create_version(agent_name=AGENT, definition=PromptAgentDefinition(
    model=model("chat"), instructions="You are J.A.R.V.I.S. Help with workshop questions. Be concise."))

conversation_ids = []
for opening, follow in [("A suit is overheating in the hangar.", "What should I do first?"),
                        ("Order me a coffee.", "Make it a double.")]:
    conv = oai.conversations.create()
    conversation_ids.append(conv.id)
    for text in (opening, follow):
        oai.responses.create(conversation=conv.id, input=text, extra_body=agent_ref(agent))
print(conversation_ids)

## 3 · Evaluate those conversations from Application Insights 🧪
No replay: the service reads the spans from Application Insights. The spans need 2–5 minutes to arrive, so the cell first waits for them with `kql_wait` (one trace per turn, as in chapter 04).

In [ ]:
has_ids = " or ".join(f'tostring(Properties) has "{c}"' for c in conversation_ids)
kql_wait(f"AppDependencies | where TimeGenerated > ago(2h) | where {has_ids} | distinct OperationId",
         hours=2, min_rows=2 * len(conversation_ids))   # 2 turns per conversation

trace_cfg = {"type": "azure_ai_source", "scenario": "traces"}
trace_criteria = [
    criterion("output_quality", "output_quality", {**whole_conversation, "tool_definitions": "{{item.tool_definitions}}"}),
    criterion("task_completion", "task_completion", whole_conversation),
    criterion("customer_satisfaction", "customer_satisfaction", whole_conversation),
]
et = oai.evals.create(name="stark-traces", data_source_config=trace_cfg, testing_criteria=trace_criteria)
rt = oai.evals.runs.create(
    eval_id=et.id, name="by-conversation-id",
    data_source={"type": "azure_ai_trace_data_source_preview",
                 "trace_source": {"type": "conversation_id_source", "conversation_ids": conversation_ids}},
    extra_body={"evaluation_level": "conversation"},
)
rt = wait_eval_run(et.id, rt.id, oai)
print(pass_rates(rt))
eval_scores(et.id, rt.id, oai)

## 4 · Sample live traffic of an agent 🧪
In production you do not know the conversation ids. Let the service pick recent ones of an agent (optionally with intelligent sampling for diversity):

In [ ]:
now = int(time.time())
run_live = oai.evals.runs.create(
    eval_id=et.id, name="agent-filter-last-24h",
    data_source={"type": "azure_ai_trace_data_source_preview",
                 "trace_source": {"type": "agent_filter", "agent_name": AGENT,
                                  "start_time": now - 24 * 3600, "end_time": now + 600, "max_traces": 5}},
    extra_body={"evaluation_level": "conversation"},
)
print("started", run_live.id, "(poll with wait_eval_run)")

## 5 · Continuous evaluation
Schedule section 4 (a nightly job, or a portal setting on the agent's **Monitor** tab) and the pass rates become a **time series**: you see quality drift when you change a prompt, a model or a tool. Pair it with alerts on Application Insights metrics (latency, error rate, tokens).

## 👀 In the portal
**Evaluation**: `stark-conversations` and `stark-traces`. The conversations they score are the ones in **Traces**.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for conv_id in conversation_ids:
    try: oai.conversations.delete(conv_id)
    except Exception: pass
for eid in (e.id, et.id):
    try: oai.evals.delete(eid)
    except Exception as exc: print("skip", eid, type(exc).__name__)
delete_agent(AGENT)
delete_dataset("stark-conversations")

## 🎯 Recap
Single answers tell you if a *response* is good. Conversations tell you if the *experience* is good. Traces tell you if *production* is good.

> 💡 Next: Act IV – the senses. [12 · OCR and Read](12_senses_ocr_read.ipynb).